In [ ]:
from ultralytics import YOLO
# 重新训练
def main():
    # 1. 加载一个预训练模型 (推荐)
    # 这里可以选择 'yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt', 'yolov8l.pt', 'yolov8x.pt'
    # 模型大小从n (nano) 到 x (extra large)，速度与精度递增。
    model = YOLO('yolov8n.pt') # 采用官方的预训练权重进行训练
    # 也可以从配置文件构建一个全新的模型: model = YOLO('yolov8n.yaml') 
    # 修改模型也可通过修改修改ultralytics的nn库和modules库，再结合结构yaml修改

    # 2. 开始训练模型
    results = model.train(
        data=r"data.yaml",
        cfg ='cfg.yaml',      
    )

    # 3. 可选: 在验证集上评估模型性能
    # metrics = model.val()

    # 4. 可选: 使用训练好的模型进行预测
    # results = model('/path/to/test/image.jpg')

if __name__ == '__main__':
    main()

In [ ]:
from ultralytics import YOLO

# 加载预训练的分割模型
model = YOLO('yolov8n.pt')  # 或 yolov8s-seg.pt, yolov8m-seg.pt 等

# 训练，冻结前 10 层
model.train(
    data=r"data.yaml",
    cfg ='cfg.yaml',  
    epochs=20,
    freeze=22,              # 冻结前22层，只训练后面的分类层，微调，这个一般面向后期数据量大后，预训练时包含要检测目标，对场景迁移应用可以微调检测头
)

In [ ]:
# 导出为onnx格式
# pip install ultralytics onnx onnxruntime
from ultralytics import YOLO
# 导出
model = YOLO('yolov8s.pt')
model.export(format='onnx', imgsz=640, batch=1, dynamic=False, opset=11, simplify=True, half=False)
print("✅ 导出完成")

In [ ]:
# 模型量化
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

model.export(
    format='engine',      # 导出为TensorRT
    imgsz=640,           # 模型训练尺寸
    int8=True,           # 启用INT8量化
    data='calibra_data.yaml', # 指向校准配置文件（500-1000张矫正图片），calibra_data.yaml在根目录下，结构个data_yaml一样
    workspace=1,         # GPU显存(GB)，
    device=0,            # GPU ID
    half=False,          # INT8与FP16二选一，此处确保为False,INT8追求的是极致的速度
    verbose=False,        # 可选，减少日志输出,
)